# Día 5 · Cuaderno 12 — APIs y extracción de datos de la web

**Programar para enseñar — Python e IA generativa para Humanidades Digitales**
Formación docente para EH1023 · Pontificia Universidad Javeriana, Facultad de Ingeniería

*(Primer bloque del Día 5 — 2 horas)*

---

Hasta ahora los datos venían en archivos que ya teníamos. Hoy aprenderás a **traer datos del mundo
real**: desde **APIs** (servicios que entregan datos) y desde **páginas web** (extracción o *scraping*).
Seguiremos el método del Día 4 —**capturar → procesar → visualizar**— ahora con IA y datos en vivo.
Al terminar podrás:

- Entender qué es una **API** y consumir una pública (sin contraseñas).
- Extraer datos de una **página web** con BeautifulSoup.
- Conocer la **ética** básica de la extracción de datos.


## 1. ¿Qué es una API?

Una **API** es una puerta por la que un servicio te entrega datos de forma ordenada, normalmente en
formato **JSON** (parecido a un diccionario de Python). Tú haces una **petición** a una dirección (URL)
y recibes datos listos para usar.

Usaremos la API pública de **Open Library** (catálogo de libros, sin necesidad de clave). Por ejemplo,
buscar libros por autor.

> 💡 En **Colab**, `requests` ya viene instalado. En local lo instalas con el monitor:
> `py -m pip install requests`.


### Capturar (con IA)
**Prompt corto:** *"En Python con la librería requests, consulta la API de Open Library
`https://openlibrary.org/search.json?author=cervantes&limit=5` y guarda la respuesta en formato JSON."*


In [ ]:
import requests

url = "https://openlibrary.org/search.json?author=cervantes&limit=5"
respuesta = requests.get(url)
datos = respuesta.json()

print("Resultados encontrados (aprox):", datos["numFound"])
print("Primeros titulos:")
for libro in datos["docs"][:5]:
    print("-", libro.get("title"))

### Procesar
**Prompt corto:** *"De esos resultados, arma una lista con el título y el primer año de publicación
(`first_publish_year`) de cada libro."*


In [ ]:
lista = []
for libro in datos["docs"][:5]:
    titulo = libro.get("title")
    anio = libro.get("first_publish_year")
    lista.append((titulo, anio))

for titulo, anio in lista:
    print(f"{anio} - {titulo}")

> 🧭 **Nota para docentes.** Una API es ideal para humanidades: bibliotecas digitales, museos y
> datos abiertos gubernamentales suelen ofrecer una. La habilidad clave no es memorizar direcciones,
> sino entender el patrón: **pedir a una URL → recibir JSON → recorrerlo como un diccionario/lista**.


## 2. Extracción de una página web (*web scraping*)

Cuando los datos **no** vienen en una API, a veces están en una **página web**. Extraerlos se llama
*web scraping*. Para practicar de forma **segura y reproducible**, usaremos una página de ejemplo del
repositorio del curso (una tabla de obras), en lugar de un sitio real.

Usaremos **BeautifulSoup**, que entiende el HTML y nos deja buscar dentro de él.

> 💡 En Colab `bs4` ya está. En local: `py -m pip install beautifulsoup4`.


### Capturar
**Prompt corto:** *"Descarga el HTML de esta URL con requests y crea un objeto BeautifulSoup."*


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://raw.githubusercontent.com/calderonf/curso-python-humanidades-digitales/main/datos/catalogo_ejemplo.html"
html = requests.get(url).text
sopa = BeautifulSoup(html, "html.parser")

print(sopa.title.text)

### Procesar
**Prompt corto:** *"De la tabla con id `catalogo`, extrae cada fila y muestra el título y el autor
(las celdas con clase `titulo` y `autor`)."*


In [ ]:
filas = sopa.select("#catalogo tbody tr")

for fila in filas:
    titulo = fila.select_one(".titulo").text
    autor = fila.select_one(".autor").text
    print(f"{titulo} — {autor}")

### Visualizar (cerrar el pipeline)
Convertimos lo extraído en una tabla de pandas y contamos por género. Así enlazamos web → datos → análisis.


In [ ]:
import pandas as pd

registros = []
for fila in sopa.select("#catalogo tbody tr"):
    registros.append({
        "titulo": fila.select_one(".titulo").text,
        "autor": fila.select_one(".autor").text,
        "anio": int(fila.select_one(".anio").text),
        "genero": fila.select_one(".genero").text,
    })

df = pd.DataFrame(registros)
print(df)
print("\nObras por genero:")
print(df["genero"].value_counts())

## 3. Ética de la extracción de datos

Extraer datos conlleva responsabilidades. Reglas básicas que conviene enseñar:

- **Revisa los términos del sitio** y el archivo `robots.txt` (indica qué se puede recorrer).
- **Prefiere las APIs** cuando existan: es la vía que el servicio ofrece para compartir datos.
- **No satures el servidor**: haz pocas peticiones y con pausas; no descargues sitios enteros.
- **Respeta los datos personales** y los derechos de autor.
- **Cita la fuente** de los datos que uses.

> 🧭 **Nota para docentes.** La dimensión ética es, quizá, lo más valioso que las humanidades aportan
> a la programación. Una gran actividad de clase es discutir *qué se debe* extraer, no solo *qué se
> puede*. Aquí el perfil humanístico de tus estudiantes es una ventaja, no una carencia.


## 4. Ejercicios graduados

Avanza a tu ritmo. **No es obligatorio llegar al reto.**

**Nivel 1 (base).** Repite la consulta a Open Library cambiando el autor por `garcia marquez`
e imprime los títulos.

**Nivel 2.** De la página de ejemplo, extrae y muestra el **año** de cada obra junto con su título.

**Reto (opcional).** Con el `df` construido por scraping, calcula cuántas obras hay **por década**
(pista: `(df["anio"] // 10) * 10`).


In [ ]:
# Tu solucion del Nivel 1:


<details>
<summary>👀 Solución posible del Nivel 1</summary>

```python
import requests
url = "https://openlibrary.org/search.json?author=garcia marquez&limit=5"
datos = requests.get(url).json()
for libro in datos["docs"][:5]:
    print(libro.get("title"))
```
</details>


In [ ]:
# Tu solucion del Nivel 2:


<details>
<summary>👀 Solución posible del Nivel 2</summary>

```python
for fila in sopa.select("#catalogo tbody tr"):
    titulo = fila.select_one(".titulo").text
    anio = fila.select_one(".anio").text
    print(f"{anio}: {titulo}")
```
</details>


In [ ]:
# Tu solucion del Reto (opcional):


<details>
<summary>👀 Solución posible del Reto</summary>

```python
df["decada"] = (df["anio"] // 10) * 10
print(df["decada"].value_counts().sort_index())
```
</details>


---

## Cierre del bloque

Ya puedes **traer datos del mundo real**: desde una **API** (JSON) y desde una **página web**
(scraping), y enlazarlos con el análisis que aprendiste esta semana, siempre con criterio **ético**.

### Esta tarde (segundo bloque del Día 5)

Cerramos el curso: una mirada a los **medios digitales** (manipulación de imágenes) y el diseño del
**proyecto integrador** para el EH1023, donde reunirás todo lo aprendido.

> Guarda tu trabajo con **Archivo → Guardar una copia en Drive**.
